In [ ]:
import os
import ollama
import chromadb
from langchain_openai import ChatOpenAI
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain_core.runnables import Runnable
from langchain_core.language_models import BaseLanguageModel
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

#download Lllama 3.2 1B GGUF
#ollama pull https://huggingface.co/bartowski/Llama-3.2-1B-Instruct-GGUF
# llm = OllamaLLM(model="huggingface.co/unsloth/Qwen3-0.6B-GGUF")
#ollama pull https://huggingface.co/bartowski/Qwen3-4B-GGUF
llm = OllamaLLM(model="huggingface.co/bartowski/Llama-3.2-1B-Instruct-GGUF")
llm_strong = OllamaLLM(model="huggingface.co/bartowski/Qwen3-4B-GGUF")

In [ ]:
def create_chain(prompt_template, model):
    prompt = PromptTemplate.from_template(prompt_template)
    return prompt | model

Zero-Shot

In [ ]:
sentiment_texts = [
    "I absolutely loved the movie! The acting was superb.",
    "The weather today is quite typical for this time of year.",
    "I'm disappointed with the service I received at the restaurant."
]

############### ZERO-SHOT
direct_task_prompt = \
"""Classify the sentiment of the following text as positive, negative, or neutral.
Do not explain your reasoning, just provide the classification in format: [sentiment]

Text: {text}

Sentiment:/no_think"""

In [ ]:
multi_step_prompt = \
"""Classify the sentiment of the following text as positive, negative, or neutral.
Provide yhour analysis in the following steps:

1. Extract relevant keywords expressing the sentiments
2. Take account into the contexts of each keywords, deciding whether they contribute to the overall sentiment
3. Make final decision based on the above analysis

Text: {text}

Analysis:"""

In [ ]:
chain = create_chain(direct_task_prompt,llm)

In [ ]:
chain.invoke({'text': sentiment_texts[0]})

In [ ]:
for text in sentiment_texts:
    res = chain.invoke({'text: text'})
    print('sentiment of {} is {}'.format(text, res))

In [ ]:
chain = create_chain(multi_step_prompt,llm)

In [ ]:
############### Few-Shot One-Task and In-Context Learning
few_shot_prompt = \
"""
    Find the word in a sentence that was reversed
    
    Examples:
    Text: I love this tcudorp! It's amazing.
    Word: tcudorp (reverse of product)
    
    Text: This eivom was terrible. I hated it.
    Word: eivom (reverse of movie)
    
    Text: The weather today si okay.
    Word: si (reverse of is)
    
    Now, work on this sentence:
    Text: {input_text}
    Word:
"""

In [ ]:
chain = create_chain(few_shot_prompt, llm)
chain.invoke({"input_text": "this class is great but no one come to the class"})

In [ ]:
zero_shot_prompt = \
"""
    Find the word in a sentence that was reversed
    
    
    Now, work on this sentence:
    Text: {input_text}
    Word:
"""

In [ ]:
chain = create_chain(zero_shot_prompt, llm)
chain.invoke({"input_text": "this class is great but no one come to the class"})

In [ ]:
zero_shot_cot_prompt = \
"""
    Find the word in a sentence that was reversed
    only output the word that you found and its reverse in a json format:
    {}
    follow step by step:

    1. 
    2. 
    3. 

    
    Now, work on this sentence:
    Text: {input_text}
    Word:
"""

In [ ]:
test_set=[
    "this class is amazing, but no one come to the class", 
    "this class is amazing, tub no one come to the class", 
    "this class is gnizama, but no one come to the class", 
]


In [ ]:
chain = create_chain(zero_shot_cot_prompt, llm)
chain.invoke({"input_text": "this class is great but no one come to the class"})

In [ ]:
############### Few-Shot Multi-Task and In-Context Learning
prompt="""
    Perform the specified task on the given text.
    
    Examples:
    Text: I love this product! It's amazing.
    Task: sentiment
    Result: Positive
    
    Text: Bonjour, comment allez-vous?
    Task: language
    Result: French
    
    Now, perform the following task:
    Text: {input_text}
    Task: {task}
    Result:
    """

In [ ]:
chain = create_chain(prompt, llm)
res = chain.invoke({'task':"sentiment", "input_text": "this class is amazion"})
print(res)

In [ ]:
############### Few-Shot Multi-Task and In-Context Learning
prompt="""
    Perform the specified task on the given text.
    
    Examples:
    Text: I love this product! It's amazing.
    Task: sentiment
    Result: Positive
    
    Text: Bonjour, comment allez-vous?
    Task: language
    Result: French
    
    Now, perform the following task:
    Text: {input_text}
    Task: {task}
    Result:
    """

In [ ]:
chain = create_chain(prompt, llm)
res = chain.invoke({'task':"translate", "input_text": "AI is amazion"})
print(res)

In [ ]:
############### Chain-of-Thought (CoT)
standard_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question concisely: {question}."
)

cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="Answer the following question step by step concisely: {question}"
)

In [ ]:
question = 'if a train reavels 120 km in 2 hours, what its average speed in km/h?'


In [ ]:
chain_standard =standard_prompt | llm
chain_cot = cot_prompt| llm

In [ ]:
chain_standard.invoke({"question": question})

In [ ]:
res = chain_cot.invoke({"question": question})
print(res)

In [ ]:
advanced_cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="""Solve the following problem step by step. For each step:
    
1. State what you're going to calculate
2. Write the formula you'll use (if applicable)
3. Perform the calculation
4. Explain the result

Question: {question}

Solution:"""
)

In [ ]:
chain_advanced_cot = advanced_cot_prompt| llm
res = chain_advanced_cot.invoke({"question": question})
print(res)

In [ ]:
############## Exercise
logical_puzzle = """In a room, there are three people: Amy, Bob, and Charlie. 
One of them always tells the truth, one always lies, and one alternates between truth and lies. 
Amy says, 'Bob is a liar.' 
Bob says, 'Charlie alternates between truth and lies.' 
Charlie says, 'Amy and I are both liars.' 
Determine the nature (truth-teller, liar, or alternator) of each person."""

In [ ]:
chain_advanced_cot = advanced_cot_prompt| llm
res = chain_advanced_cot.invoke({"question": logical_puzzle})
print(res)

In [ ]:
############## Multi-Path Reasoning
problem = """
You are planning a 3-day hiking trip for a group of 4 friends: Alice, Bob, Carla, and Daniel.
Each has different preferences and constraints:

Alice can only hike a maximum of 6 miles per day and prefers scenic routes.

Bob wants to hike at least 8 miles per day and insists on routes with water features.

Carla has a fear of steep slopes and avoids any trail marked as “strenuous.”

Daniel can only hike on days when the temperature is below 80°F.

You have access to the following trail options:

Trail Name	Miles	Difficulty	Features	Avg Temp
Eagle Rock	5	Moderate	Scenic	78°F
Crystal Falls	8	Strenuous	Waterfall	82°F
Pine Loop	6	Easy	Forest, Scenic	75°F
Canyon View	9	Moderate	Scenic, Waterfall	79°F
Sunset Ridge	7	Strenuous	Scenic	81°F

Task:
Recommend a valid 3-day itinerary that satisfies everyone’s constraints.
For each day, explain why the trail works and how it meets all the group’s needs.
If no valid itinerary exists, explain why.
"""

num_paths = 3


problem2 = "Evaluate $(1+2i)6-3i$."

In [ ]:
prompt_template = PromptTemplate(
    input_variable = ["problem", "path_number"],
    template = """slove the following problem using a unique approach.
    This is reasoning path {path_number}.

    Problem: {problem}
    Reasoning path {path_number}
    """
)

In [ ]:
paths = []
for i in range(num_paths):
    chain = prompt_template | llm
    response = chain.invoke({'problem': problem, 'path_number': str(i+1)})
    paths.append(response)

In [ ]:
len(paths)

In [ ]:
for i in enumerate(paths):
    print ("==path==",i)
    print ("========")

In [ ]:
aggre_prompt_template = PromptTemplate(
    input_variable = ["paths"],
    template = """Analyze the following reasoning paths and determine the most consistent answer.
    if there are discrepancies, explainwhy and provide the most correct answer.
    This is reasoning path {paths}.

    Most consistent answer:
    """)

In [ ]:
all_path = ""
for i in enumerate(paths):
    all_path = all_path + "Path {}:/n{}".format(i,paths)

In [ ]:
chain = aggre_prompt_template| llm
response = chain.invoke({'paths': all_path})
print(response)

In [ ]:
prompt_template = PromptTemplate(
    input_variable = ["problem", "path_number"],
    template = """slove the following problem using a unique approach.
    This is reasoning path {path_number}.

    Problem: {problem}
    Reasoning path {path_number}
    """
)

aggre_prompt_template = PromptTemplate(
    input_variable = ["paths"],
    template = """Analyze the following reasoning paths and determine the most consistent answer.
    if there are discrepancies, explainwhy and provide the most correct answer.
    This is reasoning path {paths}.

    Most consistent answer:
    """)
num_paths = 3
from tqdm.notebook import tqdm, trange
paths = []
for i in tqdm(range(num_paths)):
    chain = prompt_template | llm
    response = chain.invoke({'problem': problem2, 'path_number': str(i+1)})
    paths.append(response)



In [ ]:
len(paths)
for i in enumerate(paths):
    print ("========")
    print ("==path==",i)
    print ("========")

In [ ]:
selfcheck_prompt_template = PromptTemplate(
    input_variable = ["problem","result"],
    template = """Evaluate the consistency and reliability of the following result for the given problem.
    Problem: {problem}
    Result:{result}

    Evaluation(consider favtor like logical consistency, adherence to know facts, and potntial biases):
    """)

In [ ]:
chain = selfcheck_prompt_template | llm
response = chain.invoke({'problem': problem2, 'result': response})

In [ ]:
print(response)

In [ ]:
############ Exercise
problems = [
    "What is the capital of France?",
    # "Explain the concept of supply and demand in economics.",
    "If a train travels at 60 km/h, how long will it take to cover 180 km?"
]


In [ ]:
def exercise (problem, num_paths):
    prompt_template = PromptTemplate(
    input_variable = ["problem", "path_number"],
    template = """slove the following problem using a unique approach.
    This is reasoning path {path_number}.

    Problem: {problem}
    Reasoning path {path_number}
    """
    )
    selfcheck_prompt_template = PromptTemplate(
    input_variable = ["problem","result"],
    template = """Evaluate the consistency and reliability of the following result for the given problem.
    Problem: {problem}
    Result:{result}

    Evaluation(consider favtor like logical consistency, adherence to know facts, and potntial biases):
    """)
    
    paths = []
    for i in tqdm(range(num_paths)):
        chain = prompt_template | llm
        response = chain.invoke({'problem': problem, 'path_number': str(i+1)})
        paths.append(response)

    for i in enumerate(paths):
        chain = selfcheck_prompt_template | llm
        response = chain.invoke({'problem': problem, 'result': response})
        solution = print(response)
    return solution

In [ ]:
exercise(problems, 2)

In [ ]:

########### Task Decomposition: Break Down Complex Task
## Do it yourself


decompose_prompt_template = PromptTemplate(
    input_variable = ["task"],
    template = """Analyze the task below ad factorize it into different step-buy-step sub-task.
    Task:{task}
    output all the tasks as JSON format: [TASKNAME]: [TASK DESCRIPTION]
    """
    )
sub_prompt_template = PromptTemplate(
    input_variable = ["sub_task"],
    template = """Ado the subtask below
    Subtask: {sub_task}
    """
    )

chain = decompose_prompt_template | llm
task = "write a html file for computer science depart"
response = chain.invoke({'task':task})
print(response)


In [ ]:
import json
sub_tasks = json.loads(response)

for sub_task in sub_tasks:
    chain = sub_prompt_template | llm
    response = chain.invoke({'sub_task':sub_task["TASKDESCRIPTION"] + "/no_think"})
    print(response)